In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
#cell1 %% [code]
import subprocess, sys, os
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_wheel(pattern):
    for p in INPUT_ROOT.rglob(pattern):
        return p
    raise FileNotFoundError(f"找不到 {pattern}，请检查是否在右侧 Add Data 挂载了对应的依赖库！")

# 100% 照抄原代码：寻找并安装 onnxruntime

ONNX_WHL = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl")
if ONNX_WHL.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ONNX_WHL)], check=True)
    print("ONNX Runtime installed")
else:
    print('没有！')



# 100% 照抄原代码：寻找并降级/升级 TF 2.20 以解决死锁
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(find_wheel("tensorboard-2.20.0-*.whl"))], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(find_wheel("tensorflow-2.20.0-*.whl"))], check=True)
    print("✅ TF 2.20 installed from wheel")
except Exception as e:
    print(e)

# 导入所有后续需要的包
import onnxruntime as ort
import pandas as pd
import numpy as np
import librosa
from tqdm.auto import tqdm
import re


BASE = Path("/kaggle/input/competitions/birdclef-2026")
TRAIN_AUDIO_DIR = BASE / "train_audio"
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")

SR = 32000
WINDOW_SEC = 5
HOP_SEC = 2.5
WINDOW_SAMPLES = int(SR * WINDOW_SEC)   # 160000
HOP_SAMPLES = int(SR * HOP_SEC)         # 80000

# 1. 严格照抄原代码的 ONNX 载入方式 (Cell 4)
def find_onnx():
    for p in INPUT_ROOT.rglob("perch_v2.onnx"):
        return p
    raise FileNotFoundError("找不到 perch_v2.onnx")

ONNX_PERCH_PATH = find_onnx()

_so = ort.SessionOptions()
_so.intra_op_num_threads = 4  # 原代码指定的线程数

# 如果用 GPU 加速提特征，自动检测并启用 CUDA
providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if "CUDAExecutionProvider" in ort.get_available_providers() else ["CPUExecutionProvider"]

ONNX_SESSION = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=_so, providers=providers)
ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
ONNX_OUT_MAP = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
print(f"✅ ONNX 模型加载成功！使用的后端: {ONNX_SESSION.get_providers()[0]}")


print("环境配置严格执行完毕！")

ONNX Runtime installed
✅ TF 2.20 installed from wheel
✅ ONNX 模型加载成功！使用的后端: CPUExecutionProvider
环境配置严格执行完毕！


In [3]:
# ── Cell 2: 全局参数与文件路径配置 ──
import os
import gc
import re
import joblib
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import onnxruntime as ort

# ================= 配置区 =================
# 1. 输入数据路径
BASE_DIR = Path("/kaggle/input")
FEATURE_DIR = BASE_DIR / "notebooks/waozhang/bc2026-perch-featrue1/extracted_features"
UNLABELED_PARQUET = FEATURE_DIR / "train_soundscapes_features.parquet"

# 2. PCA与Scaler路径
MODEL_BASE = BASE_DIR / "datasets/waozhang/bc2026mlp/model"
SCALER_PATH = MODEL_BASE / "global_scaler.pkl"
PCA_PATH = MODEL_BASE / "global_pca_1024.pkl"
# /kaggle/input/datasets/waozhang/bc2026mlp/yuzhi.csv
# 3. 阈值与配置表路径
MLP_THRESHOLDS_CSV = '/kaggle/input/datasets/waozhang/bc2026mlp/yuzhi.csv'
PERCH_THRESHOLDS_CSV = BASE_DIR / "datasets/waozhang/bc2026mlp/perch_evaluation_metrics.csv"
TAXONOMY_PATH = BASE_DIR / "competitions/birdclef-2026/taxonomy.csv"

# 4. 音频与Perch相关的路径
SOUNDSCAPES_AUDIO_DIR = BASE_DIR / "competitions/birdclef-2026/train_soundscapes"
MLP_WEIGHTS_DIR = MODEL_BASE / "F1mlp"

# 5. 输出配置
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PARQUET_PATH = OUTPUT_DIR / "train_soundscapes_pseudo_labeled.parquet"
OUT_COUNTS_CSV = OUTPUT_DIR / "pseudo_label_counts.csv"
OUT_AUDIO_FILTER_CSV = OUTPUT_DIR / "high_density_audios.csv"

# 音频提取参数 (完全照抄 perch推理.py)
SR = 32_000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
PCA_DIM = 1024

# 强制使用 CPU
DEVICE = torch.device("cpu")
print(f"✅ 全局配置加载完毕，当前使用设备: {DEVICE}")

def find_file(pattern):
    for p in BASE_DIR.rglob(pattern):
        return p
    raise FileNotFoundError(f"找不到 {pattern}")

✅ 全局配置加载完毕，当前使用设备: cpu


In [4]:
# ── Cell 3: 加载待打标特征并进行 PCA 降维预处理 ──
class SpeciesMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)  # 输出原始 Logits
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

print("===== 阶段 1: 加载特征与 PCA 降维 =====")
print(f"正在读取 12 万条未标注野外环境特征: {UNLABELED_PARQUET.name}...")
df_features = pd.read_parquet(UNLABELED_PARQUET)
# 提取特征矩阵并转换为 float32
X_raw = np.stack(df_features['embedding'].values).astype(np.float32)

print("正在加载全局 Scaler 和 PCA...")
scaler = joblib.load(SCALER_PATH)
pca = joblib.load(PCA_PATH)

print("正在应用归一化与降维 (1536 -> 1024)...")
X_scaled = scaler.transform(X_raw)
X_pca = pca.transform(X_scaled).astype(np.float32)

# 将全量 PCA 特征转为 Tensor 常驻内存 (大概只需不到 500MB，非常安全)
X_tensor = torch.tensor(X_pca, dtype=torch.float32)

# 清理内存
del X_raw, X_scaled, X_pca
gc.collect()
print(f"✅ 特征降维完毕！数据尺寸: {X_tensor.shape}")

===== 阶段 1: 加载特征与 PCA 降维 =====
正在读取 12 万条未标注野外环境特征: train_soundscapes_features.parquet...
正在加载全局 Scaler 和 PCA...
正在应用归一化与降维 (1536 -> 1024)...
✅ 特征降维完毕！数据尺寸: torch.Size([127896, 1024])


In [5]:
# ── Cell 4: 解析双轨制阈值策略与 Perch 映射关系 ──
print("\n===== 阶段 2: 构建打标策略与映射字典 =====")

# 1. 载入双路阈值
df_mlp_thresh = pd.read_csv(MLP_THRESHOLDS_CSV)
df_perch_thresh = pd.read_csv(PERCH_THRESHOLDS_CSV)
taxonomy = pd.read_csv(TAXONOMY_PATH)
PRIMARY_LABELS = taxonomy["primary_label"].astype(str).tolist()

# 2. 区分高低质量 MLP (单轨信任 vs 双轨交叉验证)
# 假设 df_mlp_thresh 保证与 taxonomy 顺序一致，且带有 index, primary_label(或从文件提取), bestAUC, pavg, trust_logit, x 列
# 我们用字典来存每个物种的打标策略
strategy_dict = {}
mlp_models_list = []  # 保存对应的模型权重路径

print("正在扫描模型目录寻找对应的 .pth 权重文件...")
# 将模型权重目录下的所有文件存入字典，以便后续寻找 (文件名包含 index 和 primary_label)
pth_files = {f.name: f for f in MLP_WEIGHTS_DIR.glob("1pmlp_*.pth")}

hq_count = 0
lq_count = 0

for i, row in tqdm(df_mlp_thresh.iterrows(), total=len(df_mlp_thresh), desc="解析阈值策略"):
    idx = int(row['index'])
    # 匹配对应编号的模型文件
    model_path = next((f for name, f in pth_files.items() if name.startswith(f"1pmlp_{idx}_")), None)
    if not model_path:
        print(f"⚠️ 警告: 找不到 index {idx} 对应的 MLP 权重文件！")
        continue

    # 解析当前物种名字 (从文件名或者阈值表，根据要求提取)
    # 此处默认 primary_label 可通过 taxonomy[idx] 拿到
    sp_name = PRIMARY_LABELS[idx] 
    
    auc = row['bestAUC']
    x_val = row['x']
    
    # 构建当前物种规则字典
    strategy = {
        'index': idx,
        'primary_label': sp_name,
        'model_path': model_path,
        'mlp_trust_logit': float(row['trust_logit']),
        'mlp_pavg': float(row['pavg']),
        'is_high_quality': (auc > 0.96 and x_val > 6)
    }
    
    # 提取它的 Perch 阈值
    perch_row = df_perch_thresh[df_perch_thresh['primary_label'] == sp_name]
    if len(perch_row) > 0:
        strategy['perch_trust_logit'] = float(perch_row.iloc[0]['trust_logit'])
    else:
        strategy['perch_trust_logit'] = float('inf') # 没有基准则设为极高，防止乱打
        
    strategy_dict[idx] = strategy
    
    if strategy['is_high_quality']: hq_count += 1
    else: lq_count += 1

print(f"✅ 策略划分完成: 高质量单轨物种 {hq_count} 个，低质量双重验证物种 {lq_count} 个")

# 3. 严格还原 Perch 的词表映射逻辑
print("正在构建 Perch Logits 词表映射字典...")
ONNX_MODEL_PATH = find_file("perch_v2.onnx")
LABELS_PATH = find_file("labels.csv") # Perch的官方词表
bc_labels = pd.read_csv(LABELS_PATH).reset_index().rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
mapping = taxonomy.merge(bc_labels[["scientific_name", "bc_index"]], on="scientific_name", how="left")

direct_map = {}
proxy_map_dict = {}

for i, label in enumerate(PRIMARY_LABELS):
    row = mapping.iloc[i]
    if pd.notna(row["bc_index"]):
        direct_map[i] = int(row["bc_index"])
    else:
        sci_name = str(row["scientific_name"])
        genus = sci_name.split()[0]
        hits = bc_labels[bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)]
        if len(hits) > 0:
            proxy_map_dict[i] = hits["bc_index"].astype(int).tolist()

print(f"✅ Perch 映射字典就绪 (精确匹配: {len(direct_map)} | 代理匹配: {len(proxy_map_dict)})")


===== 阶段 2: 构建打标策略与映射字典 =====
正在扫描模型目录寻找对应的 .pth 权重文件...


解析阈值策略:   0%|          | 0/234 [00:00<?, ?it/s]

✅ 策略划分完成: 高质量单轨物种 209 个，低质量双重验证物种 25 个
正在构建 Perch Logits 词表映射字典...
✅ Perch 映射字典就绪 (精确匹配: 203 | 代理匹配: 6)


In [6]:
# strategy_dict

In [7]:
# ── Cell 5: 第一阶段 - 12万切片全量 MLP 推理 (一听) ──
print("\n===== 阶段 3: MLP 全量一听 (区分高质量录入与低质量嫌疑) =====")

# 初始化一个与 df_features 行数相同的列表，用来存放每个 5秒片段 的正样本物种名
num_rows = len(df_features)
pseudo_labels_list = [[] for _ in range(num_rows)]

# 字典：存放需要用 Perch 交叉验证的嫌疑片段 {行索引: [疑似物种1_idx, 疑似物种2_idx, ...]}
perch_suspects_by_row = {}

# 遍历 234 个物种的 MLP 模型
for sp_idx, strategy in tqdm(strategy_dict.items(), desc="MLP 向量化批量推理"):
    model_path = strategy['model_path']
    is_hq = strategy['is_high_quality']
    
    # 获取并处理阈值 (防止 NaN 报错)
    if is_hq:
        # thresh = (strategy['mlp_trust_logit'] + strategy['mlp_pavg']) / 2
        thresh = np.nanmean([strategy['mlp_trust_logit'], strategy['mlp_pavg']])
    else:
        # thresh = np.nanmax([strategy['mlp_trust_logit'], strategy['mlp_pavg']]) 
        # thresh = (strategy['mlp_trust_logit'] + strategy['mlp_pavg']) / 2
        thresh = np.nanmean([strategy['mlp_trust_logit'], strategy['mlp_pavg']])
    if pd.isna(thresh):
        continue  # 如果连阈值都没有，说明该模型严重无效，直接跳过当前物种
    
    # 动态加载对应物种的 MLP (强制分配至 CPU)
    model = SpeciesMLP(PCA_DIM)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.eval()
    
    # 一次性推完 12 万条数据！
    with torch.no_grad():
        logits = model(X_tensor).numpy()  # 得到 shape: (120000,)
        
    # 找出所有打分超过门槛的 行索引
    valid_indices = np.where(logits > thresh)[0]
    
    # 分流打标
    if is_hq:
        # 高质量物种：完全信任！直接写入最终伪标签列表
        sp_name = strategy['primary_label']
        for row_idx in valid_indices:
            pseudo_labels_list[row_idx].append(sp_name)
    else:
        # 低质量物种：列为嫌疑对象，送入下一轮 Perch 核验
        for row_idx in valid_indices:
            if row_idx not in perch_suspects_by_row:
                perch_suspects_by_row[row_idx] = []
            perch_suspects_by_row[row_idx].append(sp_idx)

print(f"✅ MLP 推理完毕！共有 {len(perch_suspects_by_row)} 个 5 秒音频切片被列为 'Perch 核查嫌疑犯'。")


===== 阶段 3: MLP 全量一听 (区分高质量录入与低质量嫌疑) =====


MLP 向量化批量推理:   0%|          | 0/234 [00:00<?, ?it/s]

✅ MLP 推理完毕！共有 34984 个 5 秒音频切片被列为 'Perch 核查嫌疑犯'。


In [8]:
# ── Cell 6: 第二阶段 - Perch 对嫌疑切片进行原始音频验证 (二听) ──
print("\n===== 阶段 4: 启动 Perch 对低质量候选片段进行二审核验 =====")

if len(perch_suspects_by_row) > 0:
    # 初始化 ONNX 引擎
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    so.inter_op_num_threads = 1
    ort_session = ort.InferenceSession(str(ONNX_MODEL_PATH), sess_options=so, providers=["CPUExecutionProvider"])
    
    input_name = ort_session.get_inputs()[0].name
    output_names = [o.name for o in ort_session.get_outputs()]
    
    # 动态探测正确的 Logits 节点 (照抄 perch推理.py)
    dummy_input = np.zeros((1, WINDOW_SAMPLES), dtype=np.float32)
    dummy_outputs = ort_session.run(None, {input_name: dummy_input})
    logits_output_name = next(output_names[i] for i, arr in enumerate(dummy_outputs) if len(arr.shape) >= 2 and arr.shape[-1] > 5000)

    # 音频加载与裁切函数
    def load_and_cut_audio(filepath, start_sec, end_sec):
        try:
            y, sr = sf.read(str(filepath), dtype="float32", always_2d=False)
            if y.ndim == 2: y = y.mean(axis=1)
            if sr != SR: y = librosa.resample(y, orig_sr=sr, target_sr=SR)
            start_frame = int(start_sec * SR)
            end_frame = int(end_sec * SR)
            y_slice = y[start_frame:end_frame]
            if len(y_slice) < WINDOW_SAMPLES:
                y_slice = np.pad(y_slice, (0, WINDOW_SAMPLES - len(y_slice)))
            else:
                y_slice = y_slice[:WINDOW_SAMPLES]
            return y_slice
        except:
            return np.zeros(WINDOW_SAMPLES, dtype=np.float32)

    # 遍历所有被列为嫌疑的音频片段行 (每段只加载音频跑 1 次，非常快)
    for row_idx, suspect_species_indices in tqdm(perch_suspects_by_row.items(), desc="Perch 二听验证"):
        
        row_data = df_features.iloc[row_idx]
        filename = row_data['filename']
        end_sec = float(row_data['end_sec'])
        start_sec = max(0.0, end_sec - WINDOW_SEC)
        filepath = SOUNDSCAPES_AUDIO_DIR / filename
        
        # 加载对应的 5s 原始音频
        y_slice = load_and_cut_audio(filepath, start_sec, end_sec)
        
        # Perch 前向传播
        x_input = np.expand_dims(y_slice, 0).astype(np.float32)
        outputs = ort_session.run([logits_output_name], {input_name: x_input})
        raw_logits = outputs[0][0]  # shape (10000+, )
        
        # 将刚刚在这段音频报警的嫌疑物种拿出来，看能不能过 Perch 最严阈值
        for sp_idx in suspect_species_indices:
            strategy = strategy_dict[sp_idx]
            sp_name = strategy['primary_label']
            perch_thresh = strategy['perch_trust_logit']
            
            # 从 Perch 的万维输出提取目标物种的打分 (精准 + 属级降级)
            if sp_idx in direct_map:
                sp_logit = raw_logits[direct_map[sp_idx]]
            elif sp_idx in proxy_map_dict:
                sp_logit = np.max(raw_logits[proxy_map_dict[sp_idx]])
            else:
                sp_logit = -100.0
                
            # 二审宣判：如果也过了 Perch 的线，则录入最终正样本库！
            if sp_logit > perch_thresh:
                pseudo_labels_list[row_idx].append(sp_name)
else:
    print("没有片段触发低质量 MLP 嫌疑，跳过 Perch 二核查。")

print("✅ 双重验证流程全部结束！")


===== 阶段 4: 启动 Perch 对低质量候选片段进行二审核验 =====


Perch 二听验证:   0%|          | 0/34984 [00:00<?, ?it/s]

✅ 双重验证流程全部结束！


In [9]:
# ── Cell 7: 组装结果并输出需求文件 ──
print("\n===== 阶段 5: 结果格式化与导出 =====")

# 1. 组装 pseudo_label 字符串
def format_labels(label_list):
    if not label_list:
        return None  # 无标签填 None
    # 去重并排序，生成分号拼接的字符串
    return ";".join(sorted(list(set(label_list))))

df_features['pseudo_label'] = [format_labels(lst) for lst in pseudo_labels_list]

# 2. 导出附带伪标签的 Parquet
print(f"1/3 正在导出带有 pseudo_label 列的 Parquet 到: {OUT_PARQUET_PATH.name}...")
df_features.to_parquet(OUT_PARQUET_PATH, index=False)

# 3. 输出统计每个物种伪标签数量的 CSV
print(f"2/3 正在生成物种数量统计报告: {OUT_COUNTS_CSV.name}...")
sp_counts = {sp: 0 for sp in PRIMARY_LABELS}

for lst in pseudo_labels_list:
    if lst:
        for sp in set(lst):
            if sp in sp_counts:
                sp_counts[sp] += 1

count_rows = []
for idx in range(len(PRIMARY_LABELS)):
    sp = PRIMARY_LABELS[idx]
    count_rows.append({
        "index": idx, 
        "primary_label": sp, 
        "pseudo_label_count": sp_counts[sp]
    })
df_counts = pd.DataFrame(count_rows)
df_counts.to_csv(OUT_COUNTS_CSV, index=False)

# 4. 输出高频包含伪标签的音频片段 (>= 8 个标签片段) 供 SSM 使用
print(f"3/3 正在提取适合 SSM 训练的连续高密音频: {OUT_AUDIO_FILTER_CSV.name}...")
df_features['has_label'] = df_features['pseudo_label'].notna()

# 分组聚合：同一个 filename 下，计算有多少行 (片段) 的 has_label 为 True
audio_group = df_features.groupby('filename')['has_label'].sum().reset_index()

# 筛选并重命名列
high_density_audios = audio_group[audio_group['has_label'] >= 8].copy()
high_density_audios.rename(columns={'has_label': 'labeled_segment_count'}, inplace=True)

# 写入文件
high_density_audios.to_csv(OUT_AUDIO_FILTER_CSV, index=False)

print("🎉 全部伪标签流水线完美执行结束！请前往 /kaggle/working/ 查看结果文件。")


===== 阶段 5: 结果格式化与导出 =====
1/3 正在导出带有 pseudo_label 列的 Parquet 到: train_soundscapes_pseudo_labeled.parquet...
2/3 正在生成物种数量统计报告: pseudo_label_counts.csv...
3/3 正在提取适合 SSM 训练的连续高密音频: high_density_audios.csv...
🎉 全部伪标签流水线完美执行结束！请前往 /kaggle/working/ 查看结果文件。
